# QSS Trading System - Master Workflow

This notebook provides an interactive interface for the complete QSS trading system workflow:
- **Section 1:** Data Curator - Manage ticker universe and data collection
- **Section 2:** SEPA Model - Build datasets and train ML models
- **Section 3:** Daily Scanner - Run scans and manage buy list

**Author:** Quantamental System  
**Last Updated:** 2024-12-05

In [ ]:
# ============================================================================
# SETUP: Import Dependencies
# ============================================================================

import sys
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

# IPython widgets for interactive UI
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from tqdm.notebook import tqdm

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Add project root to path
project_root = Path.cwd()
sys.path.append(str(project_root))

# Import project modules
import config
from src.data_engine import DataRepository
from src.database import DatabaseManager

print("✅ All dependencies loaded successfully!")
print(f"📁 Project root: {project_root}")
print(f"🔧 Config loaded from: {config.__file__}")

---
# SECTION 1: DATA CURATOR
---

## Cell 1: Get Tickers

Load ticker universe from three possible sources:
1. **Price Folder** - All tickers with cached price data
2. **FMP Screener** - Filtered universe from Financial Modeling Prep API
3. **S&P 500** - Standard & Poor's 500 index constituents

In [ ]:
# ============================================================================
# CELL 1: Get Tickers
# ============================================================================

# === USER INPUTS ===
ticker_source = widgets.Dropdown(
    options=['price_folder', 'fmp_screener', 'sp500'],
    value='price_folder',
    description='Source:',
    style={'description_width': '120px'}
)

# FMP Screener Parameters (only used if fmp_screener selected)
market_cap_min = widgets.IntText(
    value=1000000000,
    description='Min Market Cap:',
    style={'description_width': '120px'}
)

price_min = widgets.FloatText(
    value=5.0,
    description='Min Price:',
    style={'description_width': '120px'}
)

volume_min = widgets.IntText(
    value=100000,
    description='Min Volume:',
    style={'description_width': '120px'}
)

custom_tickers_input = widgets.Textarea(
    value='',
    placeholder='Enter custom tickers (comma-separated, e.g., AAPL,MSFT,TSLA)',
    description='Custom Tickers:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='600px', height='60px')
)

run_button_1 = widgets.Button(
    description='Load Tickers',
    button_style='success',
    icon='download'
)

output_1 = widgets.Output()

# Display UI
print("📊 TICKER UNIVERSE LOADER")
print("=" * 80)
display(ticker_source, market_cap_min, price_min, volume_min, custom_tickers_input, run_button_1, output_1)

# === EXECUTION LOGIC ===
loaded_tickers = []  # Global variable to store tickers

def on_load_tickers(b):
    global loaded_tickers
    with output_1:
        clear_output()
        try:
            print(f"\n🔄 Loading tickers from source: {ticker_source.value}...\n")
            
            # Check if custom tickers provided
            custom_text = custom_tickers_input.value.strip()
            if custom_text:
                loaded_tickers = [t.strip().upper() for t in custom_text.split(',') if t.strip()]
                print(f"✅ Using {len(loaded_tickers)} custom tickers")
                print(f"   Sample: {', '.join(loaded_tickers[:10])}{'...' if len(loaded_tickers) > 10 else ''}")
                return
            
            # Initialize data repository
            data_repo = DataRepository()
            
            # Map UI selection to source parameter
            source_map = {
                'price_folder': 'PRICE_FOLDER',
                'fmp_screener': 'FMP_SCREENER',
                'sp500': 'SSGA'
            }
            source = source_map[ticker_source.value]
            
            # Update FMP screener params if needed
            if ticker_source.value == 'fmp_screener':
                config.FMP_SCREENER_PARAMS.update({
                    'marketCapMoreThan': market_cap_min.value,
                    'priceMoreThan': price_min.value,
                    'volumeMoreThan': volume_min.value
                })
                print(f"   Using screener filters:")
                print(f"     Market Cap > ${market_cap_min.value:,}")
                print(f"     Price > ${price_min.value}")
                print(f"     Volume > {volume_min.value:,}")
            
            # Load tickers
            loaded_tickers = data_repo.update_universe(source=source)
            
            # Display results
            print(f"\n✅ Successfully loaded {len(loaded_tickers)} tickers")
            print(f"\n📋 Sample tickers (first 20):")
            print(f"   {', '.join(sorted(loaded_tickers[:20]))}")
            if len(loaded_tickers) > 20:
                print(f"   ... and {len(loaded_tickers) - 20} more")
            
            # Display statistics
            print(f"\n📊 Statistics:")
            print(f"   Total tickers: {len(loaded_tickers)}")
            print(f"   Source: {ticker_source.value}")
            
        except Exception as e:
            print(f"❌ Error loading tickers: {e}")
            import traceback
            traceback.print_exc()

run_button_1.on_click(on_load_tickers)

## Cell 2: Price Cache Update

Initialize or update price data cache using batch API calls.

In [ ]:
# ============================================================================
# CELL 2: Price Cache Update
# ============================================================================

# === USER INPUTS ===
use_loaded_tickers_2 = widgets.Checkbox(
    value=True,
    description='Use tickers from Cell 1',
    style={'description_width': '200px'}
)

price_source = widgets.Dropdown(
    options=['fmp', 'yfinance'],
    value='fmp',
    description='Data Source:',
    style={'description_width': '120px'}
)

min_date_price = widgets.DatePicker(
    description='Min Date:',
    value=datetime.now().date() - timedelta(days=730),  # 2 years ago
    style={'description_width': '120px'}
)

force_update_price = widgets.Checkbox(
    value=False,
    description='Force Update (re-download all)',
    style={'description_width': '200px'}
)

run_button_2 = widgets.Button(
    description='Update Price Cache',
    button_style='success',
    icon='refresh'
)

output_2 = widgets.Output()

# Display UI
print("📈 PRICE DATA CACHE UPDATER")
print("=" * 80)
display(use_loaded_tickers_2, price_source, min_date_price, force_update_price, run_button_2, output_2)

# === EXECUTION LOGIC ===
def on_update_price_cache(b):
    with output_2:
        clear_output()
        try:
            # Get tickers
            if use_loaded_tickers_2.value:
                if not loaded_tickers:
                    print("❌ No tickers loaded! Please run Cell 1 first.")
                    return
                tickers = loaded_tickers
            else:
                print("❌ Custom ticker selection not implemented. Use Cell 1 tickers.")
                return
            
            print(f"\n🔄 Updating price cache for {len(tickers)} tickers...")
            print(f"   Source: {price_source.value}")
            print(f"   Min Date: {min_date_price.value}")
            print(f"   Force Update: {force_update_price.value}\n")
            
            # Initialize data repository
            data_repo = DataRepository()
            
            # Update cache (similar to optimized_scanner.py:117)
            import time
            start_time = time.time()
            
            min_date_str = min_date_price.value.strftime('%Y-%m-%d')
            results = data_repo.update_cache(
                tickers,
                force=force_update_price.value,
                source=price_source.value,
                min_date=min_date_str
            )
            
            elapsed_time = time.time() - start_time
            
            # Count successes
            success_count = sum(results.values())
            fail_count = len(results) - success_count
            
            # Display results
            print(f"\n✅ Cache update complete!")
            print(f"\n📊 Results:")
            print(f"   Successful: {success_count}/{len(tickers)} ({success_count/len(tickers)*100:.1f}%)")
            print(f"   Failed: {fail_count}")
            print(f"   Time: {elapsed_time:.1f}s ({len(tickers)/elapsed_time:.1f} tickers/sec)")
            
            # Show failed tickers if any
            if fail_count > 0:
                failed_tickers = [t for t, success in results.items() if not success]
                print(f"\n⚠️  Failed tickers: {', '.join(failed_tickers[:10])}{'...' if len(failed_tickers) > 10 else ''}")
            
        except Exception as e:
            print(f"❌ Error updating price cache: {e}")
            import traceback
            traceback.print_exc()

run_button_2.on_click(on_update_price_cache)

## Cell 3: Fundamental Data Update

Initialize or update fundamental data (financial statements, ratios, growth metrics).

In [ ]:
# ============================================================================
# CELL 3: Fundamental Data Update
# ============================================================================

# === USER INPUTS ===
use_loaded_tickers_3 = widgets.Checkbox(
    value=True,
    description='Use tickers from Cell 1',
    style={'description_width': '200px'}
)

force_update_fund = widgets.Checkbox(
    value=False,
    description='Force Update (re-download all)',
    style={'description_width': '200px'}
)

parallel_workers_fund = widgets.IntSlider(
    value=5,
    min=1,
    max=10,
    description='Parallel Workers:',
    style={'description_width': '120px'}
)

rate_limit_fund = widgets.IntText(
    value=300,
    description='Rate Limit (req/min):',
    style={'description_width': '150px'}
)

run_button_3 = widgets.Button(
    description='Update Fundamentals',
    button_style='success',
    icon='database'
)

output_3 = widgets.Output()

# Display UI
print("📊 FUNDAMENTAL DATA UPDATER")
print("=" * 80)
display(use_loaded_tickers_3, force_update_fund, parallel_workers_fund, rate_limit_fund, run_button_3, output_3)

# === EXECUTION LOGIC ===
def on_update_fundamentals(b):
    with output_3:
        clear_output()
        try:
            # Get tickers
            if use_loaded_tickers_3.value:
                if not loaded_tickers:
                    print("❌ No tickers loaded! Please run Cell 1 first.")
                    return
                tickers = loaded_tickers
            else:
                print("❌ Custom ticker selection not implemented. Use Cell 1 tickers.")
                return
            
            print(f"\n🔄 Updating fundamental data for {len(tickers)} tickers...")
            print(f"   Parallel Workers: {parallel_workers_fund.value}")
            print(f"   Rate Limit: {rate_limit_fund.value} requests/min\n")
            
            # Import fundamental engine
            from src.fundamental_engine import FundamentalEngine
            
            # Initialize engine
            fund_engine = FundamentalEngine()
            
            # Update fundamentals with progress bar
            import time
            start_time = time.time()
            
            success_count = 0
            fail_count = 0
            
            with tqdm(total=len(tickers), desc="Updating fundamentals") as pbar:
                for ticker in tickers:
                    try:
                        fund_engine.update_ticker_fundamentals(
                            ticker,
                            force=force_update_fund.value
                        )
                        success_count += 1
                    except Exception as e:
                        fail_count += 1
                    pbar.update(1)
            
            elapsed_time = time.time() - start_time
            
            # Display results
            print(f"\n✅ Fundamental data update complete!")
            print(f"\n📊 Results:")
            print(f"   Successful: {success_count}/{len(tickers)} ({success_count/len(tickers)*100:.1f}%)")
            print(f"   Failed: {fail_count}")
            print(f"   Time: {elapsed_time:.1f}s ({len(tickers)/elapsed_time:.1f} tickers/sec)")
            
        except Exception as e:
            print(f"❌ Error updating fundamentals: {e}")
            import traceback
            traceback.print_exc()

run_button_3.on_click(on_update_fundamentals)

## Cell 4: Company Profile Update

Initialize or update company profiles (sector, industry, market cap, etc.).

In [ ]:
# ============================================================================
# CELL 4: Company Profile Update
# ============================================================================

# === USER INPUTS ===
use_loaded_tickers_4 = widgets.Checkbox(
    value=True,
    description='Use tickers from Cell 1',
    style={'description_width': '200px'}
)

force_update_profile = widgets.Checkbox(
    value=False,
    description='Force Update (re-download all)',
    style={'description_width': '200px'}
)

parallel_workers_profile = widgets.IntSlider(
    value=5,
    min=1,
    max=10,
    description='Parallel Workers:',
    style={'description_width': '120px'}
)

run_button_4 = widgets.Button(
    description='Update Profiles',
    button_style='success',
    icon='building'
)

output_4 = widgets.Output()

# Display UI
print("🏢 COMPANY PROFILE UPDATER")
print("=" * 80)
display(use_loaded_tickers_4, force_update_profile, parallel_workers_profile, run_button_4, output_4)

# === EXECUTION LOGIC ===
def on_update_profiles(b):
    with output_4:
        clear_output()
        try:
            # Get tickers
            if use_loaded_tickers_4.value:
                if not loaded_tickers:
                    print("❌ No tickers loaded! Please run Cell 1 first.")
                    return
                tickers = loaded_tickers
            else:
                print("❌ Custom ticker selection not implemented. Use Cell 1 tickers.")
                return
            
            print(f"\n🔄 Updating company profiles for {len(tickers)} tickers...")
            print(f"   Parallel Workers: {parallel_workers_profile.value}\n")
            
            # Import company profile engine
            from src.company_profile_engine import CompanyProfileEngine
            
            # Initialize engine
            profile_engine = CompanyProfileEngine()
            
            # Update profiles with progress bar
            import time
            start_time = time.time()
            
            success_count = 0
            fail_count = 0
            sector_dist = {}
            
            with tqdm(total=len(tickers), desc="Updating profiles") as pbar:
                for ticker in tickers:
                    try:
                        profile = profile_engine.update_ticker_profile(
                            ticker,
                            force=force_update_profile.value
                        )
                        if profile:
                            success_count += 1
                            sector = profile.get('sector', 'Unknown')
                            sector_dist[sector] = sector_dist.get(sector, 0) + 1
                        else:
                            fail_count += 1
                    except Exception as e:
                        fail_count += 1
                    pbar.update(1)
            
            elapsed_time = time.time() - start_time
            
            # Display results
            print(f"\n✅ Company profile update complete!")
            print(f"\n📊 Results:")
            print(f"   Successful: {success_count}/{len(tickers)} ({success_count/len(tickers)*100:.1f}%)")
            print(f"   Failed: {fail_count}")
            print(f"   Time: {elapsed_time:.1f}s ({len(tickers)/elapsed_time:.1f} tickers/sec)")
            
            # Display sector distribution
            if sector_dist:
                print(f"\n📈 Sector Distribution:")
                for sector, count in sorted(sector_dist.items(), key=lambda x: x[1], reverse=True)[:10]:
                    print(f"   {sector}: {count} ({count/success_count*100:.1f}%)")
            
        except Exception as e:
            print(f"❌ Error updating profiles: {e}")
            import traceback
            traceback.print_exc()

run_button_4.on_click(on_update_profiles)

## Cell 5: Data Health Check

Comprehensive analysis of data coverage and quality across all dimensions.

In [ ]:
# ============================================================================
# CELL 5: Data Health Check
# ============================================================================

# === USER INPUTS ===
save_health_report = widgets.Checkbox(
    value=True,
    description='Save detailed report to JSON',
    style={'description_width': '200px'}
)

health_report_path = widgets.Text(
    value='data/data_health_report.json',
    description='Report Path:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

run_button_5 = widgets.Button(
    description='Run Health Check',
    button_style='success',
    icon='heartbeat'
)

output_5 = widgets.Output()

# Display UI
print("🏥 DATA HEALTH ANALYZER")
print("=" * 80)
display(save_health_report, health_report_path, run_button_5, output_5)

# === EXECUTION LOGIC ===
def on_run_health_check(b):
    with output_5:
        clear_output()
        try:
            print(f"\n🔍 Running comprehensive data health analysis...\n")
            
            # Import data health analyzer
            from data_health_analyzer import DataHealthAnalyzer
            
            # Run full analysis
            analyzer = DataHealthAnalyzer()
            results = analyzer.run_full_analysis()
            
            # Save detailed report if requested
            if save_health_report.value:
                analyzer.save_detailed_report(health_report_path.value)
            
        except Exception as e:
            print(f"❌ Error running health check: {e}")
            import traceback
            traceback.print_exc()

run_button_5.on_click(on_run_health_check)

---
# SECTION 2: SEPA MODEL
---

## Cell 6: Build Dataset A

Generate daily feature snapshots (technical indicators + optional fundamentals).

In [ ]:
# ============================================================================
# CELL 6: Build Dataset A
# ============================================================================

# === USER INPUTS ===
start_date_a = widgets.DatePicker(
    description='Start Date:',
    value=datetime(2020, 1, 1).date(),
    style={'description_width': '120px'}
)

end_date_a = widgets.DatePicker(
    description='End Date:',
    value=datetime.now().date(),
    style={'description_width': '120px'}
)

mode_a = widgets.Dropdown(
    options=['lightweight', 'full'],
    value='lightweight',
    description='Mode:',
    style={'description_width': '120px'}
)

use_loaded_tickers_a = widgets.Checkbox(
    value=True,
    description='Use tickers from Cell 1 (uncheck for all universe)',
    style={'description_width': '350px'}
)

include_fundamentals_a = widgets.Checkbox(
    value=True,
    description='Include Fundamentals',
    style={'description_width': '200px'}
)

include_cross_sectional_a = widgets.Checkbox(
    value=False,
    description='Include Cross-Sectional Features',
    style={'description_width': '250px'}
)

n_jobs_a = widgets.IntSlider(
    value=1,
    min=-1,
    max=10,
    description='Parallel Jobs:',
    style={'description_width': '120px'}
)

skip_data_updates_a = widgets.Checkbox(
    value=True,
    description='Skip Data Updates (use existing cache)',
    style={'description_width': '300px'}
)

output_path_a = widgets.Text(
    value='data/ml/dataset_a.parquet',
    description='Output Path:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

run_button_6 = widgets.Button(
    description='Build Dataset A',
    button_style='success',
    icon='cogs'
)

output_6 = widgets.Output()

# Display UI
print("📊 DATASET A BUILDER - Daily Feature Snapshots")
print("=" * 80)
display(
    start_date_a, end_date_a, mode_a, use_loaded_tickers_a,
    include_fundamentals_a, include_cross_sectional_a, n_jobs_a,
    skip_data_updates_a, output_path_a, run_button_6, output_6
)

# === EXECUTION LOGIC ===
dataset_a = None  # Global variable to store dataset

def on_build_dataset_a(b):
    global dataset_a
    with output_6:
        clear_output()
        try:
            print(f"\n🚀 Building Dataset A...\n")
            
            # Determine tickers
            tickers_to_use = None
            if use_loaded_tickers_a.value:
                if loaded_tickers:
                    tickers_to_use = loaded_tickers
                    print(f"   Using {len(tickers_to_use)} tickers from Cell 1")
                else:
                    print(f"   ⚠️  No loaded tickers, using full universe")
            else:
                print(f"   Using full universe from FMP screener")
            
            # Import and run builder
            from build_dataset_a import build_dataset_a
            
            dataset_a = build_dataset_a(
                start_date=start_date_a.value.strftime('%Y-%m-%d'),
                end_date=end_date_a.value.strftime('%Y-%m-%d'),
                mode=mode_a.value,
                tickers=tickers_to_use,
                validate_temporal=True,
                include_fundamentals=include_fundamentals_a.value,
                include_cross_sectional=include_cross_sectional_a.value,
                n_jobs=n_jobs_a.value,
                skip_data_updates=skip_data_updates_a.value
            )
            
            if not dataset_a.empty:
                # Save dataset
                output_path = Path(output_path_a.value)
                output_path.parent.mkdir(parents=True, exist_ok=True)
                dataset_a.to_parquet(output_path, index=False, compression='snappy')
                
                # Display summary
                print(f"\n✅ Dataset A built successfully!")
                print(f"\n📊 Summary:")
                print(f"   Total rows: {len(dataset_a):,}")
                print(f"   Features: {len(dataset_a.columns) - 2}")  # Exclude date, ticker
                print(f"   Tickers: {dataset_a['ticker'].nunique()}")
                print(f"   Date range: {dataset_a['date'].min()} to {dataset_a['date'].max()}")
                print(f"   File: {output_path}")
                print(f"   Size: {output_path.stat().st_size / (1024*1024):.2f} MB")
            else:
                print("❌ No data generated!")
            
        except Exception as e:
            print(f"❌ Error building Dataset A: {e}")
            import traceback
            traceback.print_exc()

run_button_6.on_click(on_build_dataset_a)

## Cell 7: Build Dataset B

Generate trade simulation labels using historical backtesting.

In [ ]:
# ============================================================================
# CELL 7: Build Dataset B
# ============================================================================

# === USER INPUTS ===
start_date_b = widgets.DatePicker(
    description='Entry Start:',
    value=datetime(2020, 1, 1).date(),
    style={'description_width': '120px'}
)

end_date_b = widgets.DatePicker(
    description='Entry End:',
    value=datetime.now().date(),
    style={'description_width': '120px'}
)

outcome_end_b = widgets.DatePicker(
    description='Outcome End:',
    value=(datetime.now() + timedelta(days=90)).date(),
    style={'description_width': '120px'}
)

success_threshold_b = widgets.FloatSlider(
    value=15.0,
    min=5.0,
    max=30.0,
    step=1.0,
    description='Success Threshold (%):',
    style={'description_width': '150px'}
)

use_fast_simulator_b = widgets.Checkbox(
    value=True,
    description='Use Fast Vectorized Simulator (10-20x faster)',
    style={'description_width': '350px'}
)

n_jobs_b = widgets.IntSlider(
    value=1,
    min=-1,
    max=10,
    description='Parallel Jobs:',
    style={'description_width': '120px'}
)

custom_label_rule_b = widgets.Textarea(
    value='',
    placeholder='Optional: trade.return_pct >= 20 and trade.days_held <= 30',
    description='Custom Label Rule:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='600px', height='60px')
)

output_path_b = widgets.Text(
    value='data/ml/dataset_b.parquet',
    description='Output Path:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

run_button_7 = widgets.Button(
    description='Build Dataset B',
    button_style='success',
    icon='chart-line'
)

output_7 = widgets.Output()

# Display UI
print("🎯 DATASET B BUILDER - Trade Simulation Labels")
print("=" * 80)
display(
    start_date_b, end_date_b, outcome_end_b, success_threshold_b,
    use_fast_simulator_b, n_jobs_b, custom_label_rule_b,
    output_path_b, run_button_7, output_7
)

# === EXECUTION LOGIC ===
dataset_b = None  # Global variable to store dataset

def on_build_dataset_b(b):
    global dataset_b
    with output_7:
        clear_output()
        try:
            print(f"\n🚀 Building Dataset B (Trade Simulation)...\n")
            print(f"   Entry Period: {start_date_b.value} to {end_date_b.value}")
            print(f"   Outcome Window: {outcome_end_b.value}")
            print(f"   Success Threshold: {success_threshold_b.value}%")
            print(f"   Simulator: {'Fast Vectorized' if use_fast_simulator_b.value else 'Event-Driven'}\n")
            
            # Import components
            from src.data_engine import DataRepository
            from src.strategy import SEPAStrategy
            from src.features import FeatureEngineer
            from src.trading_config import TradingConfig
            
            # Initialize components
            data_repo = DataRepository()
            benchmark_data = data_repo.get_benchmark_data()
            feature_engine = FeatureEngineer(benchmark_data=benchmark_data)
            strategy = SEPAStrategy(benchmark_data=benchmark_data)
            
            # Create trading config
            labeling_function = None
            if custom_label_rule_b.value.strip():
                labeling_function = eval(f"lambda trade: 1 if ({custom_label_rule_b.value}) else 0")
            
            trading_config = TradingConfig(
                success_threshold_pct=success_threshold_b.value,
                exit_on_trend_break=True,
                exit_on_stop_loss=False,
                allow_reentry=True,
                reentry_cooldown_days=0,
                labeling_function=labeling_function
            )
            
            # Initialize simulator
            if use_fast_simulator_b.value:
                from src.trade_simulator_fast import FastTradeSimulator
                simulator = FastTradeSimulator(
                    data_repo=data_repo,
                    strategy=strategy,
                    feature_engine=feature_engine,
                    start_date=start_date_b.value.strftime('%Y-%m-%d'),
                    end_date=end_date_b.value.strftime('%Y-%m-%d'),
                    outcome_end=outcome_end_b.value.strftime('%Y-%m-%d'),
                    config=trading_config
                )
                dataset_b = simulator.run_simulation(show_progress=True, n_jobs=n_jobs_b.value)
            else:
                from src.trade_simulator import TradeSimulator
                simulator = TradeSimulator(
                    data_repo=data_repo,
                    strategy=strategy,
                    feature_engine=feature_engine,
                    start_date=start_date_b.value.strftime('%Y-%m-%d'),
                    end_date=end_date_b.value.strftime('%Y-%m-%d'),
                    outcome_end=outcome_end_b.value.strftime('%Y-%m-%d'),
                    config=trading_config
                )
                dataset_b = simulator.run_simulation()
            
            if not dataset_b.empty:
                # Save dataset
                output_path = Path(output_path_b.value)
                output_path.parent.mkdir(parents=True, exist_ok=True)
                dataset_b.to_parquet(output_path, index=False)
                
                # Get statistics
                stats = simulator.get_summary_statistics()
                
                # Display summary
                print(f"\n✅ Dataset B built successfully!")
                print(f"\n📊 Trade Statistics:")
                print(f"   Total Trades: {stats['total_trades']}")
                print(f"   Win Rate: {stats['win_rate']*100:.1f}%")
                print(f"   Avg Return: {stats['avg_return']:.2f}%")
                print(f"   Avg Days Held: {stats['avg_days_held']:.1f}")
                print(f"\n🏷️  Label Distribution:")
                for label, count in stats['label_distribution'].items():
                    label_name = "Success" if label == 1 else "Failure"
                    pct = (count / stats['total_trades']) * 100
                    print(f"   {label_name}: {count} ({pct:.1f}%)")
                print(f"\n💾 File: {output_path}")
            else:
                print("❌ No trades generated!")
            
        except Exception as e:
            print(f"❌ Error building Dataset B: {e}")
            import traceback
            traceback.print_exc()

run_button_7.on_click(on_build_dataset_b)

## Cell 8: Merge and Prepare Training Data

Merge Dataset A + B and perform validation checks.

In [ ]:
# ============================================================================
# CELL 8: Merge and Prepare Training Data
# ============================================================================

# === USER INPUTS ===
dataset_a_path_merge = widgets.Text(
    value='data/ml/dataset_a.parquet',
    description='Dataset A Path:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

dataset_b_path_merge = widgets.Text(
    value='data/ml/dataset_b.parquet',
    description='Dataset B Path:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

output_path_merge = widgets.Text(
    value='data/ml/training_dataset_final.parquet',
    description='Output Path:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

validate_temporal_merge = widgets.Checkbox(
    value=True,
    description='Validate Temporal Consistency',
    style={'description_width': '200px'}
)

report_path_merge = widgets.Text(
    value='data/ml/preparation_report.txt',
    description='Report Path:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

run_button_8 = widgets.Button(
    description='Merge Datasets',
    button_style='success',
    icon='link'
)

output_8 = widgets.Output()

# Display UI
print("🔗 DATASET MERGER - Prepare Final Training Data")
print("=" * 80)
display(
    dataset_a_path_merge, dataset_b_path_merge, output_path_merge,
    validate_temporal_merge, report_path_merge, run_button_8, output_8
)

# === EXECUTION LOGIC ===
training_dataset = None  # Global variable to store merged dataset

def on_merge_datasets(b):
    global training_dataset
    with output_8:
        clear_output()
        try:
            print(f"\n🔗 Merging datasets and validating...\n")
            
            # Import preparer
            from prepare_training_dataset import DatasetPreparer
            
            # Get date range from datasets
            # We'll use a dummy range for now (preparer will check actual range)
            preparer = DatasetPreparer(
                start_date='2020-01-01',
                end_date=datetime.now().strftime('%Y-%m-%d')
            )
            
            # Check Dataset A coverage
            print("[1/4] Checking Dataset A coverage...")
            dataset_a_stats = preparer.check_dataset_a_coverage(dataset_a_path_merge.value)
            preparer.validation_report['dataset_a'] = dataset_a_stats
            
            if not dataset_a_stats.get('exists'):
                print(f"❌ Dataset A not found: {dataset_a_path_merge.value}")
                return
            print(f"   ✅ Found {dataset_a_stats['total_rows']:,} rows, {dataset_a_stats['total_features']} features")
            
            # Check Dataset B coverage
            print("\n[2/4] Checking Dataset B coverage...")
            dataset_b_stats = preparer.check_dataset_b_coverage(dataset_b_path_merge.value)
            preparer.validation_report['dataset_b'] = dataset_b_stats
            
            if not dataset_b_stats.get('exists'):
                print(f"❌ Dataset B not found: {dataset_b_path_merge.value}")
                return
            print(f"   ✅ Found {dataset_b_stats['total_trades']:,} trades, {dataset_b_stats['win_rate']*100:.1f}% win rate")
            
            # Merge datasets
            print("\n[3/4] Merging datasets...")
            training_dataset = preparer.merge_datasets(
                dataset_a_path=dataset_a_path_merge.value,
                dataset_b_path=dataset_b_path_merge.value,
                output_path=output_path_merge.value
            )
            print(f"   ✅ Merged {len(training_dataset):,} rows")
            
            # Sanity checks
            print("\n[4/4] Running sanity checks...")
            sanity_results = preparer.perform_sanity_checks()
            preparer.validation_report['sanity_checks'] = sanity_results
            
            # Display results
            status_icon = {'PASS': '✅', 'WARNING': '⚠️', 'FAIL': '❌'}
            overall = sanity_results.get('overall_status', 'UNKNOWN')
            print(f"   {status_icon.get(overall, '❓')} Overall Status: {overall}")
            print(f"   Failures: {sanity_results.get('fail_count', 0)}")
            print(f"   Warnings: {sanity_results.get('warning_count', 0)}")
            
            # Generate and save report
            preparer.export_report(report_path_merge.value)
            
            # Final summary
            print(f"\n✅ Dataset preparation complete!")
            print(f"\n📊 Final Training Dataset:")
            print(f"   Rows: {len(training_dataset):,}")
            print(f"   Features: {len([c for c in training_dataset.columns if c not in ['date', 'ticker', 'entry_date', 'exit_date', 'label']])}")
            print(f"   Tickers: {training_dataset['ticker'].nunique()}")
            print(f"   File: {output_path_merge.value}")
            print(f"   Report: {report_path_merge.value}")
            
        except Exception as e:
            print(f"❌ Error merging datasets: {e}")
            import traceback
            traceback.print_exc()

run_button_8.on_click(on_merge_datasets)

## Cell 9: Data Preparation / Feature Engineering Framework

Template for custom data cleaning and feature engineering.

In [ ]:
# ============================================================================
# CELL 9: Data Preparation / Feature Engineering Framework
# ============================================================================

print("🔧 FEATURE ENGINEERING FRAMEWORK")
print("=" * 80)
print("\nThis cell provides a framework for custom feature engineering.")
print("Modify the code below to add your custom logic.\n")

# === CONFIGURATION ===
training_dataset_path_fe = 'data/ml/training_dataset_final.parquet'
output_path_fe = 'data/ml/training_dataset_cleaned.parquet'

# === LOAD DATASET ===
print("[1/5] Loading training dataset...")
df_fe = pd.read_parquet(training_dataset_path_fe)
print(f"   Loaded {len(df_fe):,} rows, {len(df_fe.columns)} columns\n")

# === HANDLE MISSING VALUES ===
print("[2/5] Handling missing values...")
print(f"   Missing values before: {df_fe.isnull().sum().sum():,} ({df_fe.isnull().sum().sum()/df_fe.size*100:.2f}%)")

# TODO: Add your custom missing value handling here
# Example:
# df_fe['feature_name'] = df_fe['feature_name'].fillna(df_fe['feature_name'].median())

print(f"   Missing values after: {df_fe.isnull().sum().sum():,} ({df_fe.isnull().sum().sum()/df_fe.size*100:.2f}%)\n")

# === FEATURE SCALING / NORMALIZATION ===
print("[3/5] Feature scaling / normalization...")

# TODO: Add your custom scaling logic here
# Example:
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# df_fe[numeric_cols] = scaler.fit_transform(df_fe[numeric_cols])

print("   Scaling complete\n")

# === FEATURE CREATION ===
print("[4/5] Creating custom features...")

# TODO: Add your custom feature creation here
# Example:
# df_fe['custom_feature'] = df_fe['feature1'] / df_fe['feature2']

print(f"   Total features: {len(df_fe.columns)}\n")

# === SAVE CLEANED DATASET ===
print("[5/5] Saving cleaned dataset...")
output_path = Path(output_path_fe)
output_path.parent.mkdir(parents=True, exist_ok=True)
df_fe.to_parquet(output_path, index=False)
print(f"   ✅ Saved to: {output_path}")
print(f"   Size: {output_path.stat().st_size / (1024*1024):.2f} MB\n")

print("✅ Feature engineering complete!")

## Cell 10: Define Fold and Training Parameters

Configure temporal splitting and feature selection parameters.

In [ ]:
# ============================================================================
# CELL 10: Define Fold and Training Parameters
# ============================================================================

# === USER INPUTS ===
dataset_path_train = widgets.Text(
    value='data/ml/training_dataset_final.parquet',
    description='Dataset Path:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

purge_gap_days = widgets.IntSlider(
    value=60,
    min=30,
    max=120,
    step=10,
    description='Purge Gap (days):',
    style={'description_width': '150px'}
)

correlation_threshold = widgets.FloatSlider(
    value=0.95,
    min=0.70,
    max=0.99,
    step=0.01,
    description='Correlation Threshold:',
    style={'description_width': '150px'}
)

top_n_features = widgets.IntText(
    value=0,
    description='Top N Features (0=all):',
    style={'description_width': '180px'}
)

precision_k_pct = widgets.FloatSlider(
    value=0.2,
    min=0.1,
    max=0.5,
    step=0.1,
    description='Precision@k:',
    style={'description_width': '150px'}
)

optimize_hyperparameters = widgets.Checkbox(
    value=False,
    description='Optimize Hyperparameters (slow)',
    style={'description_width': '250px'}
)

n_optuna_trials = widgets.IntSlider(
    value=50,
    min=10,
    max=100,
    step=10,
    description='Optuna Trials:',
    style={'description_width': '150px'}
)

run_button_10 = widgets.Button(
    description='Prepare Training',
    button_style='success',
    icon='cog'
)

output_10 = widgets.Output()

# Display UI
print("⚙️ TRAINING CONFIGURATION")
print("=" * 80)
display(
    dataset_path_train, purge_gap_days, correlation_threshold, top_n_features,
    precision_k_pct, optimize_hyperparameters, n_optuna_trials,
    run_button_10, output_10
)

# === EXECUTION LOGIC ===
train_config = {}  # Global variable to store configuration

def on_prepare_training(b):
    global train_config
    with output_10:
        clear_output()
        try:
            print(f"\n🔧 Preparing training configuration...\n")
            
            # Import preparation modules
            from src.model_preparation import prepare_training_data
            
            # Prepare data
            top_n = top_n_features.value if top_n_features.value > 0 else None
            
            prep_result = prepare_training_data(
                dataset_path=dataset_path_train.value,
                purge_gap_days=purge_gap_days.value,
                correlation_threshold=correlation_threshold.value,
                keep_top_n=top_n
            )
            
            # Store configuration
            train_config = {
                'df': prep_result['df'],
                'folds': prep_result['folds'],
                'splitter': prep_result['splitter'],
                'selector': prep_result['selector'],
                'optimize': optimize_hyperparameters.value,
                'n_trials': n_optuna_trials.value,
                'precision_k': precision_k_pct.value
            }
            
            # Display summary
            print(f"✅ Training preparation complete!\n")
            print(f"📊 Configuration:")
            print(f"   Dataset: {len(train_config['df']):,} rows")
            print(f"   Temporal Folds: {len(train_config['folds'])}")
            print(f"   Selected Features: {len(train_config['selector'].selected_features)}")
            print(f"   Dropped Features: {sum(len(v) for v in train_config['selector'].dropped_features.values())}")
            print(f"   Purge Gap: {purge_gap_days.value} days")
            print(f"   Optimize Hyperparameters: {optimize_hyperparameters.value}")
            if optimize_hyperparameters.value:
                print(f"   Optuna Trials: {n_optuna_trials.value}")
            print(f"   Precision@k: {int(precision_k_pct.value*100)}%")
            
            # Display fold structure
            print(f"\n📅 Fold Structure:")
            for fold in train_config['folds']:
                fold_type = "PRODUCTION" if fold.get('is_production', False) else "Validation"
                print(f"   Fold {fold['fold_id']} ({fold_type}):")
                print(f"     Train: {fold['train_start']} to {fold['train_end']} ({fold['train_size']:,} rows)")
                if not fold.get('is_production', False):
                    print(f"     Test:  {fold['test_start']} to {fold['test_end']} ({fold['test_size']:,} rows)")
            
        except Exception as e:
            print(f"❌ Error preparing training: {e}")
            import traceback
            traceback.print_exc()

run_button_10.on_click(on_prepare_training)

## Cell 11: Train Model

Train XGBoost models for each temporal fold.

In [ ]:
# ============================================================================
# CELL 11: Train Model
# ============================================================================

# === USER INPUTS ===
model_output_dir = widgets.Text(
    value='models',
    description='Model Output Dir:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

eval_output_dir = widgets.Text(
    value='evaluation',
    description='Eval Output Dir:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px')
)

use_gpu = widgets.Checkbox(
    value=False,
    description='Use GPU (if available)',
    style={'description_width': '200px'}
)

run_button_11 = widgets.Button(
    description='Train Models',
    button_style='success',
    icon='rocket'
)

output_11 = widgets.Output()

# Display UI
print("🚀 MODEL TRAINING")
print("=" * 80)
display(model_output_dir, eval_output_dir, use_gpu, run_button_11, output_11)

# === EXECUTION LOGIC ===
trained_models = []  # Global variable to store trained models

def on_train_models(b):
    global trained_models
    with output_11:
        clear_output()
        try:
            # Check if training config exists
            if not train_config:
                print("❌ No training configuration! Please run Cell 10 first.")
                return
            
            print(f"\n🚀 Training {len(train_config['folds'])} fold(s)...\n")
            
            # Import trainer
            from src.train_model import SEPAModelTrainer
            
            # Create output directories
            Path(model_output_dir.value).mkdir(parents=True, exist_ok=True)
            Path(eval_output_dir.value).mkdir(parents=True, exist_ok=True)
            
            trained_models = []
            
            for fold_idx, fold in enumerate(train_config['folds']):
                fold_type = "PRODUCTION" if fold.get('is_production', False) else "Validation"
                print(f"\n{'='*80}")
                print(f"TRAINING FOLD {fold['fold_id']} ({fold_type})")
                print(f"{'='*80}\n")
                
                # Get train/test data
                X_train, X_test, y_train, y_test = train_config['splitter'].get_fold_data(
                    train_config['df'],
                    fold_idx=fold_idx,
                    feature_columns=train_config['selector'].selected_features
                )
                
                # Apply feature selection
                X_train = train_config['selector'].transform(X_train)
                X_test = train_config['selector'].transform(X_test)
                
                # Split train/val for validation folds
                if fold.get('is_production', False):
                    X_train_split = X_train
                    y_train_split = y_train
                    X_val_split = None
                    y_val_split = None
                    print(f"Production fold: Using ALL training data ({len(X_train_split)} rows)\n")
                else:
                    train_size = int(len(X_train) * 0.8)
                    X_train_split = X_train.iloc[:train_size]
                    y_train_split = y_train.iloc[:train_size]
                    X_val_split = X_train.iloc[train_size:]
                    y_val_split = y_train.iloc[train_size:]
                    print(f"Train: {len(X_train_split)}, Val: {len(X_val_split)}, Test: {len(X_test)} rows\n")
                
                # Initialize trainer
                trainer = SEPAModelTrainer(precision_k_pct=train_config['precision_k'])
                
                # Optimize hyperparameters (only on first fold)
                if train_config['optimize'] and fold_idx == 0:
                    print(f"Optimizing hyperparameters ({train_config['n_trials']} trials)...")
                    best_params = trainer.optimize_hyperparameters(
                        X_train_split,
                        y_train_split,
                        X_val_split,
                        y_val_split,
                        n_trials=train_config['n_trials']
                    )
                    print(f"Best parameters found!\n")
                elif fold_idx > 0 and train_config['optimize']:
                    print("Reusing hyperparameters from Fold 1\n")
                    trainer.best_params = trained_models[0].best_params
                
                # Train model
                print("Training final model...")
                model = trainer.train_with_best_params(
                    X_train_split,
                    y_train_split,
                    X_val_split,
                    y_val_split
                )
                
                # Save model
                trainer.save_model(model_output_dir.value, fold_id=fold['fold_id'])
                trained_models.append(trainer)
                
                print(f"\n✅ Fold {fold['fold_id']} training complete!")
            
            print(f"\n{'='*80}")
            print(f"✅ ALL MODELS TRAINED SUCCESSFULLY!")
            print(f"{'='*80}")
            print(f"\n📁 Models saved to: {Path(model_output_dir.value).resolve()}")
            print(f"   Total models: {len(trained_models)}")
            
        except Exception as e:
            print(f"❌ Error training models: {e}")
            import traceback
            traceback.print_exc()

run_button_11.on_click(on_train_models)

## Cell 12: Review Results and Select Best Model

Compare fold performance and select the best model for production.

In [ ]:
# ============================================================================
# CELL 12: Review Results and Select Best Model
# ============================================================================

print("📊 MODEL EVALUATION & SELECTION")
print("=" * 80)
print("\nThis cell evaluates all trained folds and helps you select the best model.\n")

# Check if models are trained
if not trained_models:
    print("❌ No trained models found! Please run Cell 11 first.")
else:
    # Import evaluator
    from src.evaluate_model import ModelEvaluator
    
    eval_dir = eval_output_dir.value
    evaluator = ModelEvaluator(output_dir=eval_dir)
    
    # Evaluate each fold (skip production folds)
    all_fold_results = []
    
    for fold_idx, fold in enumerate(train_config['folds']):
        if fold.get('is_production', False):
            print(f"Skipping Fold {fold['fold_id']} (PRODUCTION - no test data)\n")
            continue
        
        print(f"Evaluating Fold {fold['fold_id']}...")
        
        # Get test data
        X_train, X_test, y_train, y_test = train_config['splitter'].get_fold_data(
            train_config['df'],
            fold_idx=fold_idx,
            feature_columns=train_config['selector'].selected_features
        )
        
        X_test = train_config['selector'].transform(X_test)
        test_indices = fold['test_indices']
        df_test = train_config['df'].loc[test_indices]
        
        # Get trained model
        model = trained_models[fold_idx].model
        
        # Evaluate
        fold_results = evaluator.evaluate_fold(
            model=model,
            X_test=X_test,
            y_test=y_test,
            df_test=df_test,
            fold_id=fold['fold_id'],
            k_values=[0.1, 0.2, 0.3]
        )
        
        all_fold_results.append(fold_results)
    
    # Generate comprehensive report
    evaluator.generate_report(
        all_fold_results,
        output_path=f'{eval_dir}/evaluation_report.json'
    )
    
    # Create comparison table
    comparison_data = []
    for result in all_fold_results:
        comparison_data.append({
            'Fold': result['fold_id'],
            'Precision@10%': result['metrics']['precision_at_k'].get(0.1, 0),
            'Precision@20%': result['metrics']['precision_at_k'].get(0.2, 0),
            'Precision@30%': result['metrics']['precision_at_k'].get(0.3, 0),
            'Win Rate': result['backtest']['win_rate'],
            'Avg Return': result['backtest']['avg_return'],
            'AUC-ROC': result['metrics']['auc_roc']
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    
    print(f"\n✅ Evaluation complete!\n")
    print("📊 FOLD COMPARISON:")
    print("=" * 80)
    display(comparison_df)
    
    # Visualizations
    print(f"\n📈 Visualizations saved to: {eval_dir}/")
    print(f"   - ROC curves: roc_curve_fold_*.png")
    print(f"   - Feature importance: feature_importance_fold_*.png")
    print(f"   - Full report: evaluation_report.json")
    
    # Selection helper
    print(f"\n🎯 MODEL SELECTION HELPER:")
    print("=" * 80)
    
    best_by_precision = comparison_df.loc[comparison_df['Precision@20%'].idxmax()]
    best_by_return = comparison_df.loc[comparison_df['Avg Return'].idxmax()]
    
    print(f"\n   Best by Precision@20%: Fold {best_by_precision['Fold']} ({best_by_precision['Precision@20%']:.3f})")
    print(f"   Best by Avg Return: Fold {best_by_return['Fold']} ({best_by_return['Avg Return']:.2f}%)")
    
    print(f"\n💡 Recommendation: Review the evaluation report and plots before selecting.")
    print(f"   Consider consistency across folds and alignment with your strategy goals.\n")

## Cell 13: Retrain Production Model

Retrain selected model on ALL available data for production deployment.

In [ ]:
# ============================================================================
# CELL 13: Retrain Production Model
# ============================================================================

# === USER INPUTS ===
best_fold_id = widgets.IntText(
    value=1,
    description='Best Fold ID:',
    style={'description_width': '150px'}
)

use_all_data_prod = widgets.Checkbox(
    value=True,
    description='Use ALL data (no train/test split)',
    style={'description_width': '250px'}
)

production_model_path = widgets.Text(
    value='models/model_production.json',
    description='Production Model:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='500px')
)

run_button_13 = widgets.Button(
    description='Train Production Model',
    button_style='danger',
    icon='check-circle'
)

output_13 = widgets.Output()

# Display UI
print("🏭 PRODUCTION MODEL TRAINING")
print("=" * 80)
display(best_fold_id, use_all_data_prod, production_model_path, run_button_13, output_13)

# === EXECUTION LOGIC ===
production_model = None  # Global variable

def on_train_production(b):
    global production_model
    with output_13:
        clear_output()
        try:
            # Check if models exist
            if not trained_models:
                print("❌ No trained models! Please run Cell 11 first.")
                return
            
            print(f"\n🏭 Training production model...\n")
            print(f"   Using hyperparameters from Fold {best_fold_id.value}")
            print(f"   Training on ALL data: {use_all_data_prod.value}\n")
            
            # Get best model's hyperparameters
            fold_idx = best_fold_id.value - 1  # 0-indexed
            best_trainer = trained_models[fold_idx]
            
            # Import trainer
            from src.train_model import SEPAModelTrainer
            
            # Initialize new trainer with best params
            prod_trainer = SEPAModelTrainer(precision_k_pct=train_config['precision_k'])
            prod_trainer.best_params = best_trainer.best_params
            
            # Prepare data
            if use_all_data_prod.value:
                # Use ALL data (no split)
                feature_cols = train_config['selector'].selected_features
                X_all = train_config['df'][feature_cols]
                y_all = train_config['df']['label']
                
                # Apply feature selection
                X_all = train_config['selector'].transform(X_all)
                
                print(f"   Training on {len(X_all):,} total rows\n")
                
                # Train
                production_model = prod_trainer.train_with_best_params(
                    X_all,
                    y_all,
                    None,  # No validation data
                    None
                )
            else:
                print("❌ Non-all-data training not implemented. Use all data for production.")
                return
            
            # Save production model
            prod_path = Path(production_model_path.value)
            prod_path.parent.mkdir(parents=True, exist_ok=True)
            production_model.save_model(str(prod_path))
            
            print(f"\n✅ Production model trained successfully!\n")
            print(f"📊 Model Info:")
            print(f"   Features: {len(train_config['selector'].selected_features)}")
            print(f"   Training Rows: {len(X_all):,}")
            print(f"   Hyperparameters: Inherited from Fold {best_fold_id.value}")
            print(f"   Model Path: {prod_path}")
            print(f"   Model Version: {datetime.now().strftime('%Y%m%d_%H%M%S')}")
            
            print(f"\n🚀 Ready for deployment in scanner (use --use-ml flag)!\n")
            
        except Exception as e:
            print(f"❌ Error training production model: {e}")
            import traceback
            traceback.print_exc()

run_button_13.on_click(on_train_production)

---
# SECTION 3: DAILY SCANNER
---

## Cell 14: Define Scanner Scope

Configure and run the daily scanner (single day or date range).

In [ ]:
# ============================================================================
# CELL 14: Define Scanner Scope
# ============================================================================

# === USER INPUTS ===
scan_mode = widgets.Dropdown(
    options=['single_day', 'date_range'],
    value='single_day',
    description='Scan Mode:',
    style={'description_width': '120px'}
)

scan_date_single = widgets.DatePicker(
    description='Scan Date:',
    value=datetime.now().date(),
    style={'description_width': '120px'}
)

start_date_range = widgets.DatePicker(
    description='Start Date:',
    value=(datetime.now() - timedelta(days=30)).date(),
    style={'description_width': '120px'}
)

end_date_range = widgets.DatePicker(
    description='End Date:',
    value=datetime.now().date(),
    style={'description_width': '120px'}
)

use_ml_scanner = widgets.Checkbox(
    value=False,
    description='Enable ML Scoring',
    style={'description_width': '150px'}
)

model_path_scanner = widgets.Text(
    value='models/model_production.json',
    description='ML Model:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px')
)

csv_output_scanner = widgets.Checkbox(
    value=False,
    description='Export to CSV',
    style={'description_width': '150px'}
)

debug_mode_scanner = widgets.Checkbox(
    value=False,
    description='Debug Mode (detailed metrics)',
    style={'description_width': '200px'}
)

scanner_tickers_input = widgets.Textarea(
    value='',
    placeholder='Optional: specific tickers (comma-separated)',
    description='Tickers:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='500px', height='60px')
)

run_button_14 = widgets.Button(
    description='Run Scanner',
    button_style='success',
    icon='search'
)

output_14 = widgets.Output()

# Display UI
print("🔍 DAILY SCANNER")
print("=" * 80)
display(
    scan_mode, scan_date_single, start_date_range, end_date_range,
    use_ml_scanner, model_path_scanner, csv_output_scanner,
    debug_mode_scanner, scanner_tickers_input, run_button_14, output_14
)

# === EXECUTION LOGIC ===
def on_run_scanner(b):
    with output_14:
        clear_output()
        try:
            # Parse tickers
            scanner_tickers = None
            if scanner_tickers_input.value.strip():
                scanner_tickers = [t.strip().upper() for t in scanner_tickers_input.value.split(',') if t.strip()]
            
            # Import scanner
            from optimized_scanner import run_optimized_scanner
            
            if scan_mode.value == 'single_day':
                print(f"\n🔍 Running single-day scan for {scan_date_single.value}...\n")
                
                run_optimized_scanner(
                    scan_date=scan_date_single.value.strftime('%Y-%m-%d'),
                    csv_output=csv_output_scanner.value,
                    use_ml=use_ml_scanner.value,
                    model_path=model_path_scanner.value if use_ml_scanner.value else None,
                    tickers=scanner_tickers,
                    debug=debug_mode_scanner.value
                )
            else:
                print(f"\n🔍 Running date-range scan from {start_date_range.value} to {end_date_range.value}...\n")
                print("⚠️  Date range mode uses 2D vectorized scanning (faster)\n")
                
                # Import components for date range mode
                import sys
                from datetime import datetime, timedelta
                import pandas as pd
                import time
                
                # Simulate command-line args
                class Args:
                    def __init__(self):
                        self.date_range = [
                            start_date_range.value.strftime('%Y-%m-%d'),
                            end_date_range.value.strftime('%Y-%m-%d')
                        ]
                        self.use_ml = use_ml_scanner.value
                        self.model_path = model_path_scanner.value if use_ml_scanner.value else None
                
                args = Args()
                
                # Run date range scan (from optimized_scanner.py:1186-1307)
                from src.data_engine import DataRepository
                from src.features import FeatureEngineer
                from src.vectorized_screening import VectorizedSEPAScreener
                
                start_date = datetime.strptime(args.date_range[0], '%Y-%m-%d')
                end_date = datetime.strptime(args.date_range[1], '%Y-%m-%d')
                
                print("[PRE-SCAN] Updating cache...")
                data_repo = DataRepository()
                tickers = data_repo.update_universe()
                min_date = (pd.Timestamp.now() - pd.DateOffset(years=2)).strftime('%Y-%m-%d')
                results = data_repo.update_cache(tickers, force=False, source='fmp', min_date=min_date)
                
                print("[PRE-SCAN] Loading price data...")
                all_ticker_data = data_repo.get_batch_data(tickers)
                
                print("\n[Step 1/4] Computing features...")
                benchmark_data = data_repo.get_benchmark_data()
                feature_engine = FeatureEngineer(benchmark_data=benchmark_data)
                enriched_data = feature_engine.process_universe_batch(all_ticker_data)
                
                print("[Step 2/4] Building 2D matrix...")
                lookback_days = 251
                data_start_date = start_date - timedelta(days=lookback_days)
                df_matrix = VectorizedSEPAScreener.build_2d_matrix(
                    enriched_data,
                    start_date=pd.Timestamp(data_start_date),
                    end_date=pd.Timestamp(end_date)
                )
                
                print("[Step 3/4] Computing SEPA status...")
                df_matrix = VectorizedSEPAScreener.add_sepa_status_column(df_matrix)
                
                print("[Step 4/4] Finding signal transitions...")
                buy_signals, sell_signals = VectorizedSEPAScreener.find_signal_transitions(
                    df_matrix,
                    date_range_start=pd.Timestamp(start_date),
                    date_range_end=pd.Timestamp(end_date)
                )
                
                print(f"\n✅ Date range scan complete!")
                print(f"   Buy signals: {len(buy_signals)}")
                print(f"   Sell signals: {len(sell_signals)}")
            
        except Exception as e:
            print(f"❌ Error running scanner: {e}")
            import traceback
            traceback.print_exc()

run_button_14.on_click(on_run_scanner)

## Cell 15: Review Buy List

Display and analyze the current buy list from the database.

In [ ]:
# ============================================================================
# CELL 15: Review Buy List
# ============================================================================

# === USER INPUTS ===
as_of_date_buylist = widgets.DatePicker(
    description='As Of Date:',
    value=datetime.now().date(),
    style={'description_width': '120px'}
)

active_only_buylist = widgets.Checkbox(
    value=True,
    description='Active Only',
    style={'description_width': '120px'}
)

sort_by_buylist = widgets.Dropdown(
    options=['signal_date', 'ml_rank', 'rs', 'price_change_%'],
    value='ml_rank',
    description='Sort By:',
    style={'description_width': '120px'}
)

export_csv_buylist = widgets.Checkbox(
    value=False,
    description='Export to CSV',
    style={'description_width': '120px'}
)

run_button_15 = widgets.Button(
    description='Load Buy List',
    button_style='success',
    icon='list'
)

output_15 = widgets.Output()

# Display UI
print("📋 BUY LIST VIEWER")
print("=" * 80)
display(as_of_date_buylist, active_only_buylist, sort_by_buylist, export_csv_buylist, run_button_15, output_15)

# === EXECUTION LOGIC ===
def on_load_buy_list(b):
    with output_15:
        clear_output()
        try:
            print(f"\n📋 Loading buy list...\n")
            
            # Initialize database
            db = DatabaseManager()
            
            # Get buy list
            buy_list_df = db.get_buy_list(
                active_only=active_only_buylist.value,
                as_of_date=as_of_date_buylist.value.strftime('%Y-%m-%d')
            )
            
            if buy_list_df.empty:
                print("No active buy signals found.")
                return
            
            # Calculate price changes
            if 'signal_price' in buy_list_df.columns and 'current_price' in buy_list_df.columns:
                buy_list_df['price_change_$'] = buy_list_df['current_price'] - buy_list_df['signal_price']
                buy_list_df['price_change_%'] = ((buy_list_df['current_price'] - buy_list_df['signal_price']) / buy_list_df['signal_price'] * 100)
            
            # Sort
            if sort_by_buylist.value in buy_list_df.columns:
                buy_list_df = buy_list_df.sort_values(sort_by_buylist.value)
            
            # Display summary
            print(f"✅ Buy List loaded ({len(buy_list_df)} signals)\n")
            print(f"📊 Summary Statistics:")
            
            if 'signal_date' in buy_list_df.columns:
                buy_list_df['days_on_list'] = (pd.Timestamp(as_of_date_buylist.value) - pd.to_datetime(buy_list_df['signal_date'])).dt.days
                print(f"   Avg days on list: {buy_list_df['days_on_list'].mean():.1f}")
                print(f"   Newest signal: {buy_list_df['signal_date'].max()}")
                print(f"   Oldest signal: {buy_list_df['signal_date'].min()}")
            
            if 'price_change_%' in buy_list_df.columns:
                print(f"\n💰 Performance:")
                print(f"   Avg return: {buy_list_df['price_change_%'].mean():.2f}%")
                print(f"   Best performer: {buy_list_df.loc[buy_list_df['price_change_%'].idxmax(), 'ticker']} (+{buy_list_df['price_change_%'].max():.2f}%)")
                print(f"   Worst performer: {buy_list_df.loc[buy_list_df['price_change_%'].idxmin(), 'ticker']} ({buy_list_df['price_change_%'].min():.2f}%)")
            
            # Display dataframe
            print(f"\n📋 BUY LIST:\n")
            display_cols = ['ticker', 'signal_date', 'signal_price', 'current_price', 'price_change_%',
                           'rs', 'volume_ratio', 'ml_probability', 'ml_rank']
            available_cols = [col for col in display_cols if col in buy_list_df.columns]
            display(buy_list_df[available_cols].head(50))
            
            # Export if requested
            if export_csv_buylist.value:
                output_path = Path('data/scanner_output') / f'buy_list_{as_of_date_buylist.value}.csv'
                output_path.parent.mkdir(parents=True, exist_ok=True)
                buy_list_df.to_csv(output_path, index=False)
                print(f"\n💾 Exported to: {output_path}")
            
        except Exception as e:
            print(f"❌ Error loading buy list: {e}")
            import traceback
            traceback.print_exc()

run_button_15.on_click(on_load_buy_list)

## Cell 16: Database Management Examples

Demonstrate common database operations and queries.

In [ ]:
# ============================================================================
# CELL 16: Database Management Examples
# ============================================================================

print("🗄️ DATABASE MANAGEMENT")
print("=" * 80)
print("\nThis cell provides examples of common database operations.\n")

# Initialize database
db = DatabaseManager()

print("Available operations:")
print("  1. Inspect buy_list table")
print("  2. Inspect buy_list_activity log")
print("  3. Clear future signals (temporal consistency)")
print("  4. Export tables to CSV")
print("  5. Custom SQL query\n")

# === OPERATION 1: Inspect buy_list ===
print("[1] BUY LIST TABLE:")
print("=" * 80)
buy_list = db.get_buy_list(active_only=False)
print(f"Total entries: {len(buy_list)}")
print(f"Active: {len(buy_list[buy_list['is_active'] == 1]) if 'is_active' in buy_list.columns else 'N/A'}")
if not buy_list.empty:
    display(buy_list.head(10))
else:
    print("(Empty table)")

# === OPERATION 2: Inspect activity log ===
print("\n[2] BUY LIST ACTIVITY LOG:")
print("=" * 80)
# Export to temporary file to view
temp_activity_path = 'data/temp_activity.csv'
db.export_to_csv('buy_list_activity', temp_activity_path)
activity_df = pd.read_csv(temp_activity_path)
print(f"Total activity records: {len(activity_df)}")
if not activity_df.empty:
    print(f"\nRecent activity:")
    display(activity_df.tail(10))
else:
    print("(No activity logged)")

# === OPERATION 3: Clear future signals (example) ===
print("\n[3] CLEAR FUTURE SIGNALS (Example - NOT EXECUTED):")
print("=" * 80)
print("To clear signals after a specific date:")
print("  deleted = db.clear_future_signals('2024-01-01')")
print("  print(f'Cleared {deleted[\"buy_list_deleted\"]} signals')")

# === OPERATION 4: Export tables ===
print("\n[4] EXPORT TABLES TO CSV:")
print("=" * 80)
export_dir = Path('data/db_exports')
export_dir.mkdir(parents=True, exist_ok=True)

tables_to_export = ['buy_list', 'buy_list_activity']
for table in tables_to_export:
    export_path = export_dir / f'{table}_{datetime.now().strftime("%Y%m%d")}.csv'
    try:
        db.export_to_csv(table, str(export_path))
        print(f"  ✅ {table}: {export_path}")
    except Exception as e:
        print(f"  ❌ {table}: {e}")

# === OPERATION 5: Custom SQL query (example) ===
print("\n[5] CUSTOM SQL QUERY (Example):")
print("=" * 80)
print("Example query to find top ML-ranked stocks:")
print("")
print("SELECT ticker, ml_rank, ml_probability, signal_date")
print("FROM buy_list")
print("WHERE is_active = 1 AND ml_rank IS NOT NULL")
print("ORDER BY ml_rank ASC")
print("LIMIT 10;")
print("")
print("To execute:")
print("  result = db.execute_query('SELECT ...')")

print("\n✅ Database management operations complete!")